# FreeControl — smoke (load + reference-control + stock path)
Loads `remyxai/freecontrol-flux-modular`, runs structural control (astronaut structure + new prompt) and the
`reference_image=None` stock path. Runtime: 80GB A100, FLUX.1-dev license.

In [ ]:
import subprocess
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image

In [ ]:
import torch, os, numpy as np
os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="1"
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
REPO="remyxai/freecontrol-flux-modular"; H=W=1024

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained(REPO, trust_remote_code=True)
assert type(pipe.blocks).__name__ == "FreeControlBlock", type(pipe.blocks).__name__
pipe.load_components(dtype=DT); pipe.to(DEV)   # base FLUX.1-dev ~36GB fits 80GB
def get(o):
    o = o.images if hasattr(o,"images") else (o.get("images") if isinstance(o,dict) else o)
    return o[0] if isinstance(o,(list,tuple)) else o
print("loaded:", type(pipe.blocks).__name__)

In [ ]:
from PIL import Image
from skimage import data
from IPython.display import display
REF = Image.fromarray(data.astronaut()).convert("RGB").resize((W,H))
img = get(pipe(prompt="a bronze statue bust, museum, dramatic lighting", reference_image=REF,
               structure_strength=0.3, num_inference_steps=20, height=H, width=W, output_type="pil", output="images"))
assert img.size==(W,H), img.size
display(Image.fromarray(np.concatenate([np.asarray(REF.resize((384,384))), np.asarray(img.resize((384,384)))],1)))
print("[control] OK — reference structure + prompt content:", img.size)
stock = get(pipe(prompt="a bowl of fruit", reference_image=None, num_inference_steps=20, height=H, width=W,
                 output_type="pil", output="images"))
assert stock.size==(W,H)
print("[stock] reference_image=None path OK:", stock.size, "-> SMOKE COMPLETE")